In [5]:
# Import libraries
import torch
import torch.nn as nn
import torch.nn.functional as functional
import torch.optim as optim

# Setup hyperparameters
batch_size = 32
seq_len = 8
max_iters = 3000
eval_interval = 300
eval_iters = 200
learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
embed_dim = 32

# Setup fixed seed for reproduceablity
torch.manual_seed(1337)


In [2]:
torch.cuda.is_available()

True

In [7]:
# Download the tiny shakespear dataset which contains the entire work of shakespear
!wget 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'

--2025-05-01 18:14:36--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.008s  

2025-05-01 18:14:37 (140 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [8]:
# Read the dataset
with open('input.txt', 'r') as f:
    data = f.read()
    print(f'Number of characters in text dataset: {len(data)}')

Number of characters in text dataset: 1115394


In [9]:
# Generate vocabulary (Set of all possible tokens)
vocab = sorted(list(set(data)))
vocab_size = len(vocab)
print(f'Length of Vocabulary: {len(vocab)}')
print(f'Vocabulary: {"".join(vocab)}')

Length of Vocabulary: 65
Vocabulary: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [10]:
# Tokenization (Converting raw string of text into tokens)
stoi = {ch: idx for idx,ch in enumerate(vocab)}
itos = {idx: ch for ch, idx in stoi.items()}
# Converts raw input text to tokens
encode = lambda input_str: [stoi[ch] for ch in input_str]
# Convert the integer tokens into readable strings
decode = lambda input_tokens: "".join([itos[idx] for idx in input_tokens])

print(encode('hii there'))
print(decode(encode('hii there')))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [11]:
# Encode tiny shakespear using the above character level tokenizer
data_t = torch.tensor(encode(data), dtype=torch.int64)
print(data_t[:1000])

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
        47, 59, 57,  1, 47, 57,  1, 41, 

In [12]:
# Train/valid split
n = int(0.90 * len(data_t))
train_t = data_t[:n]
valid_t = data_t[n:]
print(f'Number of characters for training: {len(train_t)}')
print(f'Number of characters for validation: {len(valid_t)}')


Number of characters for training: 1003854
Number of characters for validation: 111540


In [13]:
# EDA of the training data
# seq_len = maximum length of input that the LLM can take as input (input + output)
print(f'First training input: {train_t[:8]}')
print(f'First training output corresponding to each position: {train_t[1:seq_len+1]}')

for idx in range(seq_len):
    x = train_t[:idx+1]
    y = train_t[idx+1]
    print(f'When input is {x}, output is {y}')

First training input: tensor([18, 47, 56, 57, 58,  1, 15, 47])
First training output corresponding to each position: tensor([47, 56, 57, 58,  1, 15, 47, 58])
When input is tensor([18]), output is 47
When input is tensor([18, 47]), output is 56
When input is tensor([18, 47, 56]), output is 57
When input is tensor([18, 47, 56, 57]), output is 58
When input is tensor([18, 47, 56, 57, 58]), output is 1
When input is tensor([18, 47, 56, 57, 58,  1]), output is 15
When input is tensor([18, 47, 56, 57, 58,  1, 15]), output is 47
When input is tensor([18, 47, 56, 57, 58,  1, 15, 47]), output is 58


In [14]:
# Dataloader (to generate batch)
torch.manual_seed(1337)
seq_len = 8 # Determines the maximum context (input len) that the model can see to make prediction. Also called time dimension
batch_size = 4 # Deteremines the number of independent seqeunces that will be processed in one pass

def get_batch(split):
    data = train_t if split == 'train' else valid_t
    sample_idxs = torch.randint(0, len(data) - seq_len, size=(batch_size,))
    x = torch.stack([data[start_idx:start_idx+seq_len] for start_idx in sample_idxs])
    y = torch.stack([data[start_idx+1:start_idx+seq_len+1] for start_idx in sample_idxs])
    x = x.to(device)
    y = y.to(device)
    return x,y

x, y = get_batch('train') # Returens data in the dimensions: (Batch, Time) (B,T)
print(x)
print(y)

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]], device='cuda:0')
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]], device='cuda:0')


In [23]:
@torch.no_grad()
def estimate_losses():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters, device=device)
        for step in range(eval_iters):
            x, y = get_batch(split)
            x, y = x.to(device), y.to(device)
            logits, loss = model(x, y)
            losses[step] = loss.item()
        out[split] = losses.mean().to('cpu')
    model.train()
    return out

In [16]:
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Embedding layer in pytorch takes as input (B,T) dimension and returns the (B,T,C) (Batch, Time, Channel) dimension
        # It is basically a lookup table
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.positional_embedding = nn.Embedding(seq_len, embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size)
        return

    def forward(self, inputs, targets=None):
        assert inputs.ndim == 2, "Incorrect number of dimesions as input to forward() method"

        token_embedding = self.token_embedding(inputs) # logits = (B, T, C)
        positions = torch.arange(inputs.shape[1])
        position_embedding = self.positional_embedding(positions)
        embedding = token_embedding + position_embedding
        logits = self.lm_head(embedding)

        if targets is None:
            loss = None
        else:
            # Since the tokens along time dimension within one seqeucne are not interacting with each other, we can simply make predictions on each token indiviaully
            # by flatenning out the matrix
            logits = logits.view(-1, logits.shape[-1]) # Convert to (B*T, C) shape
            targets = targets.view(-1)
            # cross_entropy() loss function accepts predictions as raw unnormalized logits for each output class.
            # It accepts the target as either the correct class index for each sample in batch for the same shape with the probability of class individually defined
            loss = functional.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, input_batch, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(input_batch)
            # Select only last token of each sequence / time dimension
            logits = logits[:, -1, :]

            # Generate probability distribution on the last token of each sequence
            prob = functional.softmax(logits, dim=1)
            # Takes as input (B, PROB_DIST) and returns (B, num_samples). This means it returns num_samples sampled from each prob_distribution.
            predictions = torch.multinomial(prob, num_samples=1)
            input_batch = torch.cat((input_batch, predictions), dim=1)
        return input_batch

In [12]:
model = BigramLanguageModel()
model.to(device)
inputs, targets = get_batch(split='train')
logits, loss = model(inputs, targets)
print(f'Logits shape: {logits.shape}')
print(f'Loss shape: {loss.shape}')
print(loss)

Logits shape: torch.Size([32, 65])
Loss shape: torch.Size([])
tensor(4.4367, grad_fn=<NllLossBackward0>)


In [13]:
inputs = torch.zeros(1,1, dtype=torch.int64)
max_new_tokens = 7
output = model.generate(inputs, max_new_tokens)
print(decode(output[0].tolist()))


pJQO,So


In [ ]:
# Train the model
model = BigramLanguageModel()
optim = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for step in range(max_iters):

    if step % eval_interval == 0:
        losses = estimate_losses()
        print(f'Training Loss: {losses["train"]}\nValidation Loss: {losses["val"]}')

    inputs, targets = get_batch(split='train')
    logits, loss = model(inputs, targets)
    optim.zero_grad()
    loss.backward()
    optim.step()

In [ ]:
inputs = torch.zeros(1, 1, dtype=torch.int64, device=device)
outputs = model.generate(inputs, max_new_tokens=7)
print(decode(outputs[0].tolist()))

In [ ]:
# Trick to implement self attention through matrix multiplication
# Here we use a basic form of attention called aggreation where we take the average representation of each token till that token

B,T,C = 4, 8, 2 # Batch, Time, Channel dimension
x = torch.randn(B, T, C)
print(x[0])

# First implementation, inefficient due to looping
xbow = torch.zeros(B, T, C)
for b in range(B):
    for t in range(T):
        xbow[b, t] = x[b, :t+1].mean(dim=0)
print(xbow[0])


In [ ]:
# Using matrix multiplication for averaging
mask = torch.ones(T, T)
mask = torch.tril(mask)
mask = mask / mask.sum(dim=1, keepdim=True)
xbow2 = mask @ x
print('Mask=')
print(mask)
print('xAveraged=')
print(xbow2[0])

In [ ]:
# Now implementing self attention (averaging) using matrix multipication
# Second implementation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(dim=1, keepdim=True)
print('Weight=')
print(wei)

# Now, since our batch was B x T x C, we will be performing BMM (Batched Matrix Multiply). mask will be broadcasted
xbow3 = wei @ x
print(xbow3[0], xbow[0])

In [ ]:
# Third implementation (Using softmax logic)
mask = torch.tril(torch.ones(T, T))
wei = torch.zeros(T, T)
# masked_fill function takes a mask of boolean values and fills only the positions that are True
wei = wei.masked_fill(mask == 0, float('-inf'))
print(wei)
wei = functional.softmax(wei, dim=1)
print(wei)
xbow4 = wei @ x
print(xbow4[0])

In [ ]:
# Self Attention built from above logic
B,T,C = 4,8,32
head_size = 16

# Each input token will emit three vectors (Query, Key, Value)
# Query: What is this token looking for?
# Key: What informat can this token provide
# Value: Information that the token will share with other tokens
query_layer = nn.Linear(C, head_size)    # B,T,head_size
key_layer = nn.Linear(C, head_size)      # B,T,head_size
value_layer = nn.Linear(C, head_size)    # B,T,head_size

data = torch.randn(B, T, C)
print(f'------ Input Data ------\n{data[0]}')
query = query_layer(data)
key = key_layer(data)
value = value_layer(data)

wei = query @ key.transpose(-1,-2)
# We need to divide by sqrt(head_size) because the variance for each
# entry in the wei matrix changes from 1 to head_size. So to keep the std
# deviation of each entry in wei matrix, we need to divide by sqrt(head_size)
# The effect of this is that all the values in the wei matrix are not very far
# off from each other. Because if one entry becomes very large and rest are
# relatively small, then when we apply softmax to this wei matrix, it becomes
# kind of like one-hot matrix where each row will have one entry with very large
# value and rest will have very small (negligible) value. This will cause the
# attention mechanism to focus only on one token and ignore information from the
# rest of the tokens. Thus to diffues the values in the rows properly so that the
# scores (wei) are not took peaky so that all tokens have some resonable amount
# of contribution.
wei = wei / (torch.tensor(head_size) ** (1/2))
mask = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(mask == 0, float('-inf'))
print(f'----- Unnormalized Raw Attention Scores -----\n{wei[0]}')
wei_n = nn.functional.softmax(wei, dim=-1)
print(f'----- Normalized Attention Scores ------\n{wei_n[0]}')
output_data = wei_n @ value
print(f'----- Output Data ------\n{output_data[0]}')

In [54]:
class AttentionHead(nn.Module):
  def __init__(self, n_seq, n_embed, head_size):
    super().__init__()
    self.query = nn.Linear(n_embed, head_size, bias=False)
    self.key = nn.Linear(n_embed, head_size, bias=False)
    self.value = nn.Linear(n_embed, head_size, bias=False)
    self.register_buffer('mask', torch.tril(torch.ones(n_seq, n_seq)) == 0)

  def forward(self, input):
    n_batch, n_seq, n_embed = input.shape
    query = self.query(input)
    key = self.key(input)
    value = self.value(input)
    n_head = query.shape[-1]

    scores = (query @ key.transpose(-1, -2)) / (n_head ** (1/2))
    scores = scores.masked_fill(self.mask[:n_seq, :n_seq], float('-inf'))
    scores = nn.functional.softmax(scores, dim=-1)

    output = scores @ value
    return output

class MultiHeadAttention(nn.Module):
  def __init__(self, n_seq, n_embed, num_heads, head_size):
   super().__init__()
   assert num_heads * head_size == n_embed, 'Num Heads * Head Size should be equal to Embedding dimension size'
   self.multihead_attention = nn.ModuleList([AttentionHead(n_seq, n_embed, head_size) for _ in range(num_heads)])

  def forward(self, x):
    output = torch.cat([self_attention(x) for self_attention in self.multihead_attention], dim=-1)
    return output

class FeedForward(nn.Module):
  def __init__(self, n_embed):
    super().__init__()
    self.ffn = nn.Sequential(nn.Linear(n_embed, n_embed), nn.ReLU())

  def forward(self, x):
    output = self.ffn(x)
    return output

class BasicGPT(nn.Module):
  def __init__(self, n_seq, n_embed, num_heads):
    super().__init__()
    self.n_seq, self.n_embed, self.num_heads = n_seq, n_embed, num_heads
    self.token_embedding = nn.Embedding(vocab_size, n_embed)
    self.position_embedding = nn.Embedding(n_seq, n_embed)
    self.multihead_attention = MultiHeadAttention(n_seq, n_embed, num_heads, n_embed // num_heads)
    self.ffn = FeedForward(n_embed)
    self.lm_head = nn.Linear(n_embed, vocab_size)

  def forward(self, input, target=None):
    n_batch, n_seq = input.shape
    token_embedding = self.token_embedding(input)
    position_embedding = self.position_embedding(torch.arange(n_seq))
    logits = token_embedding + position_embedding
    logits = self.multihead_attention(logits)
    logits = self.ffn(logits)
    logits = self.lm_head(logits)

    loss = None
    if target is not None:
      logits = logits.view(-1, vocab_size)
      target = target.view(-1)
      loss = nn.functional.cross_entropy(logits, target)

    return logits,loss

  def generate(self, input, max_new_tokens):
    self.eval()
    input_copy = input.detach().clone()
    for _ in range(max_new_tokens):
      input = input[:, -self.n_seq:]
      logits, loss = self(input)
      logits = logits[:, -1, :]
      logits = nn.functional.softmax(logits, dim=-1)
      sampled_idx = torch.multinomial(logits, 1)
      input = torch.cat((input, sampled_idx), dim=1)
      input_copy = torch.cat((input_copy, sampled_idx), dim=1)
    self.train()
    return input_copy



In [55]:
# Setup hyperparameters
batch_size = 32
seq_len = 8
max_iters = 5000
eval_interval = 500
eval_iters = 200
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
embed_dim = 32
num_heads = 4

# Train the model
model = BasicGPT(seq_len, embed_dim, num_heads)
optim = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for step in range(max_iters):
    if step % eval_interval == 0:
        losses = estimate_losses()
        print(f'Training Loss: {losses["train"]}\nValidation Loss: {losses["val"]}')

    inputs, targets = get_batch(split='train')
    logits, loss = model(inputs, targets)
    optim.zero_grad()
    loss.backward()
    optim.step()

Training Loss: 4.153298377990723
Validation Loss: 4.1537299156188965
Training Loss: 2.600736379623413
Validation Loss: 2.608760118484497
Training Loss: 2.4705591201782227
Validation Loss: 2.459819793701172
Training Loss: 2.3878836631774902
Validation Loss: 2.39359974861145
Training Loss: 2.355581283569336
Validation Loss: 2.362075090408325
Training Loss: 2.3218634128570557
Validation Loss: 2.343308925628662
Training Loss: 2.2946410179138184
Validation Loss: 2.305169105529785
Training Loss: 2.27583646774292
Validation Loss: 2.289168357849121
Training Loss: 2.2585813999176025
Validation Loss: 2.285101890563965
Training Loss: 2.2437288761138916
Validation Loss: 2.270756721496582


In [56]:
inputs = torch.zeros(1, 1, dtype=torch.int64, device=device)
outputs = model.generate(inputs, max_new_tokens=300)
# print(decode(outputs[0].tolist()))
for output in outputs:
  print(decode(output.tolist()))


BO wom cher.

KICENCH Enot proo thor ning weak alll fureold; nole I sallate
Lond camyir and eele Ind, the mold sas andot, dand whimsh.
Satupen aglill of ursamed of ark, icersces knore bethin.
CI HANCD:
O, to ner srelp:or
Till ter:
All line, math
yelot but's acher amen of toe frater sagsemime;

And t


In [49]:
@torch.no_grad()
def estimate_losses():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters, device=device)
        for step in range(eval_iters):
            x, y = get_batch(split)
            x, y = x.to(device), y.to(device)
            logits, loss = model(x, y)
            losses[step] = loss.item()
        out[split] = losses.mean().to('cpu')
    model.train()
    return out

In [50]:
class AttentionHead(nn.Module):
  def __init__(self, n_seq, n_embed, head_size, dropout=0.2):
    super().__init__()
    self.query = nn.Linear(n_embed, head_size, bias=False)
    self.key = nn.Linear(n_embed, head_size, bias=False)
    self.value = nn.Linear(n_embed, head_size, bias=False)
    self.register_buffer('mask', torch.tril(torch.ones(n_seq, n_seq)) == 0)
    self.dropout = nn.Dropout(dropout)

  def forward(self, input):
    n_batch, n_seq, n_embed = input.shape
    query = self.query(input)
    key = self.key(input)
    value = self.value(input)
    n_head = query.shape[-1]

    scores = (query @ key.transpose(-1, -2)) / (n_head ** (1/2))
    scores = scores.masked_fill(self.mask[:n_seq, :n_seq], float('-inf'))
    scores = nn.functional.softmax(scores, dim=-1)
    scores = self.dropout(scores)

    output = scores @ value
    return output

class MultiHeadAttention(nn.Module):
  def __init__(self, n_seq, n_embed, num_heads, head_size, dropout=0.2):
   super().__init__()
   assert num_heads * head_size == n_embed, 'Num Heads * Head Size should be equal to Embedding dimension size'
   self.multihead_attention = nn.ModuleList([AttentionHead(n_seq, n_embed, head_size) for _ in range(num_heads)])
   self.proj = nn.Linear(n_embed, n_embed)
   self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    output = torch.cat([self_attention(x) for self_attention in self.multihead_attention], dim=-1)
    output = self.proj(output)
    output = self.dropout(output)
    return output

class FeedForward(nn.Module):
  def __init__(self, n_embed, dropout=0.2):
    super().__init__()
    self.ffn = nn.Sequential(nn.Linear(n_embed, 4 * n_embed),
                             nn.ReLU(),
                             nn.Linear(4 * n_embed, n_embed),
                             nn.Dropout(dropout))

  def forward(self, x):
    output = self.ffn(x)
    return output

class Block(nn.Module):
  def __init__(self, n_seq, n_embed, num_heads):
    super().__init__()
    head_size = n_embed // num_heads
    self.multihead_attention = MultiHeadAttention(n_seq, n_embed, num_heads, head_size)
    self.ffn = FeedForward(n_embed)
    self.layernorm1 = nn.LayerNorm(n_embed)
    self.layernorm2 = nn.LayerNorm(n_embed)

  def forward(self, x):
    output = x + self.multihead_attention(self.layernorm1(x))
    output = x + self.ffn(self.layernorm2(output))
    return output

class BasicGPT(nn.Module):
  def __init__(self, n_seq, num_blocks, n_embed, num_heads):
    super().__init__()
    self.n_seq, self.n_embed, self.num_heads = n_seq, n_embed, num_heads
    self.token_embedding = nn.Embedding(vocab_size, n_embed)
    self.position_embedding = nn.Embedding(n_seq, n_embed)
    self.blocks = nn.Sequential(*[Block(n_seq, n_embed, num_heads) for _ in range(num_blocks)])
    self.layernorm = nn.LayerNorm(n_embed)
    self.lm_head = nn.Linear(n_embed, vocab_size)

  def forward(self, input, target=None):
    n_batch, n_seq = input.shape
    token_embedding = self.token_embedding(input)
    position_embedding = self.position_embedding(torch.arange(n_seq, device=device))
    logits = token_embedding + position_embedding
    logits = self.blocks(logits)
    logits = self.layernorm(logits)
    logits = self.lm_head(logits)

    loss = None
    if target is not None:
      logits = logits.view(-1, vocab_size)
      target = target.view(-1)
      loss = nn.functional.cross_entropy(logits, target)

    return logits,loss

  def generate(self, input, max_new_tokens):
    self.eval()
    input_copy = input.detach().clone()
    for _ in range(max_new_tokens):
      input = input[:, -self.n_seq:]
      logits, loss = self(input)
      logits = logits[:, -1, :]
      logits = nn.functional.softmax(logits, dim=-1)
      sampled_idx = torch.multinomial(logits, 1)
      input = torch.cat((input, sampled_idx), dim=1)
      input_copy = torch.cat((input_copy, sampled_idx), dim=1)
    self.train()
    return input_copy



In [ ]:
# Setup hyperparameters
batch_size = 64
seq_len = 256
max_iters = 5000
eval_interval = 500
eval_iters = 200
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
embed_dim = 384
num_heads = 6
num_blocks = 6
dropout = 0.2
if torch.cuda.is_available(): print(f'---- Using CUDA Device ----')

# Train the model
model = BasicGPT(seq_len, num_blocks, embed_dim, num_heads)
model.to(device)
optim = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for step in range(max_iters):
    if step % eval_interval == 0:
        losses = estimate_losses()
        print(f'Iteration: {step}\nTraining Loss: {losses["train"]}\nValidation Loss: {losses["val"]}')

    inputs, targets = get_batch(split='train')
    logits, loss = model(inputs, targets)
    optim.zero_grad()
    loss.backward()
    optim.step()

---- Using CUDA Device ----
Iteration: 0
Training Loss: 4.346076488494873
Validation Loss: 4.335861682891846
Iteration: 500
Training Loss: 1.9392367601394653
Validation Loss: 2.0409674644470215


In [67]:
inputs = torch.zeros(1, 1, dtype=torch.int64, device=device)
outputs = model.generate(inputs, max_new_tokens=300)
# print(decode(outputs[0].tolist()))
for output in outputs:
  print(decode(output.tolist()))


Thous age is were brey:
Turgaun:
Dere wood; sidvigh:
Keld hav.' whithen him all in of hast:
Unecures sus hene hood.

MERW,-
So put tancien.

DKAPUS:
Bereash.

SOM
To't thus tongigh ousue
iThen do sir,
That but day of intizeng.

ROCENCEMBERLONCENCENTER:
Wats
him man word, tie blay lord sues of lurses
